In [2]:
import pandas as pd
import sqlite3
import os

In [3]:
# 1) Load raw CSV files 
patients = pd.read_csv("../data/patients.csv")
print("patient shape: ", patients.shape)

visits = pd.read_csv("../data/visits.csv")
print("visits shape: ", visits.shape)

billing = pd.read_csv("../data/billing.csv")
print("billing shape: ", billing.shape)

patient shape:  (5000, 7)
visits shape:  (25000, 8)
billing shape:  (25000, 7)


In [4]:
# 2) Create sqlite db

os.makedirs("../db", exist_ok=True)

conn = sqlite3.connect("../db/hospital.db")
cursor = conn.cursor()
print("Database connected: ", conn)

Database connected:  <sqlite3.Connection object at 0x130258b80>


In [5]:
# 3) Load dataframes into sqlite tables

patients.to_sql("patients", conn, if_exists="replace", index=False)
visits.to_sql("visits", conn, if_exists="replace", index=False)
billing.to_sql("billing", conn, if_exists="replace", index=False)
print("All three tables loaded into hospital db")

All three tables loaded into hospital db


In [8]:
# 4) Preview tables

print("=== PATIENTS (first 3 rows) ===")
print(pd.read_sql("select * from patients limit 3",conn).to_string())

print("=== VISITS (first 3 rows) ===")
print(pd.read_sql("select * from visits limit 3", conn).to_string())

print("=== BILLING (first 3 rows) ===")
print(pd.read_sql("select * from billing limit 3", conn).to_string())

=== PATIENTS (first 3 rows) ===
   patient_id  age gender       city insurance_provider  chronic_flag registration_date
0           1   53      M  Hyderabad         SecureLife             0        2025-05-14
1           2   42      M       Pune         HealthPlus             0        2025-11-18
2           3   56      F  Hyderabad         HealthPlus             0        2025-05-11
=== VISITS (first 3 rows) ===
   visit_id  patient_id  visit_date   department visit_type  length_of_stay_hours risk_score  doctor_id
0         1         756  2025-10-18   Cardiology         ER                  3.48        Low        169
1         2        4102  2025-04-06  Orthopedics        OPD                 15.31       High        148
2         3        2964  2025-07-13          ICU         ER                 34.36        Low        153
=== BILLING (first 3 rows) ===
   bill_id  visit_id  billed_amount  approved_amount claim_status  payment_days billing_date
0        1         1       23577.37           

In [10]:
# 5) Deparment workload analysis

dep_workload = pd.read_sql("""
SELECT 
    department,
    COUNT(visit_id)                         AS total_visits,
    ROUND(AVG(length_of_stay_hours), 2)     AS avg_los_hours,
    ROUND(MAX(length_of_stay_hours), 2)     AS max_los_hours,
    SUM(CASE WHEN risk_score = 'High' 
        THEN 1 ELSE 0 END)                  AS high_risk_visits
FROM visits 
GROUP BY department
ORDER BY total_visits DESC
""", conn)

print(dep_workload.to_string(index=False))

 department  total_visits  avg_los_hours  max_los_hours  high_risk_visits
    General          4228          19.43          78.42               839
         ER          4220          19.53          74.81               872
  Neurology          4165          19.72          70.12               846
Orthopedics          4164          19.66          71.60               842
 Cardiology          4159          19.60          77.42               790
        ICU          4064          19.36          72.47               845


In [11]:
# 6) Doctor Risk Cases
# Which doctors handle the most high-risk patients?

doctor_risk = pd.read_sql("""
    SELECT 
        doctor_id,
        COUNT(visit_id)                              AS total_visits,
        SUM(CASE WHEN risk_score = 'High' 
            THEN 1 ELSE 0 END)                       AS high_risk_cases,
        ROUND(
            100.0 * SUM(CASE WHEN risk_score = 'High' 
            THEN 1 ELSE 0 END) / COUNT(visit_id), 1) AS high_risk_pct
    FROM visits
    GROUP BY doctor_id
    ORDER BY high_risk_cases DESC
    LIMIT 10
""", conn)

print("Top 10 Doctors by High Risk Cases")
print(doctor_risk.to_string(index=False))

Top 10 Doctors by High Risk Cases
 doctor_id  total_visits  high_risk_cases  high_risk_pct
       174           264               71           26.9
       198           252               69           27.4
       169           279               68           24.4
       177           266               67           25.2
       135           261               65           24.9
       105           250               65           26.0
       188           285               64           22.5
       180           290               64           22.1
       131           266               62           23.3
       178           245               61           24.9


In [13]:
# 7) Patient Visit Patterns
# How many visits does each patient make on average?

visit_patterns = pd.read_sql("""
    SELECT 
        visit_frequency_bucket,
        COUNT(patient_id) AS num_patients
    FROM (
        SELECT 
            patient_id,
            COUNT(visit_id) AS visit_count,
            CASE 
                WHEN COUNT(visit_id) = 1     THEN '1 visit'
                WHEN COUNT(visit_id) BETWEEN 2 AND 3 THEN '2-3 visits'
                WHEN COUNT(visit_id) BETWEEN 4 AND 5 THEN '4-5 visits'
                ELSE '6+ visits'
            END AS visit_frequency_bucket
        FROM visits
        GROUP BY patient_id
    )
    GROUP BY visit_frequency_bucket
    ORDER BY num_patients DESC
""", conn)

print("Patient Visit Frequency Distribution")
print(visit_patterns.to_string(index=False))

Patient Visit Frequency Distribution
visit_frequency_bucket  num_patients
             6+ visits          1937
            4-5 visits          1731
            2-3 visits          1150
               1 visit           149


In [14]:
# 8) Risk Score Distribution by Department
risk_by_dept = pd.read_sql("""
    SELECT 
        department,
        SUM(CASE WHEN risk_score = 'Low'    THEN 1 ELSE 0 END) AS low,
        SUM(CASE WHEN risk_score = 'Medium' THEN 1 ELSE 0 END) AS medium,
        SUM(CASE WHEN risk_score = 'High'   THEN 1 ELSE 0 END) AS high,
        COUNT(*) AS total
    FROM visits
    GROUP BY department
    ORDER BY high DESC
""", conn)

print("Risk Score Distribution by Department")
print(risk_by_dept.to_string(index=False))

Risk Score Distribution by Department
 department  low  medium  high  total
         ER 2092    1256   872   4220
  Neurology 2058    1261   846   4165
        ICU 2037    1182   845   4064
Orthopedics 2078    1244   842   4164
    General 2123    1266   839   4228
 Cardiology 2082    1287   790   4159


## Financial Analytics

In [15]:
# 9) Insurance Billing Breakdown
# Revenue and claim outcomes by insurance provider.
insurance_billing = pd.read_sql("""
    SELECT 
        p.insurance_provider,
        COUNT(b.bill_id)                         AS total_claims,
        ROUND(SUM(b.billed_amount), 0)           AS total_billed,
        ROUND(AVG(b.billed_amount), 0)           AS avg_billed,
        ROUND(SUM(b.approved_amount), 0)         AS total_approved,
        SUM(CASE WHEN b.claim_status = 'Paid'     
            THEN 1 ELSE 0 END)                   AS paid,
        SUM(CASE WHEN b.claim_status = 'Pending'  
            THEN 1 ELSE 0 END)                   AS pending,
        SUM(CASE WHEN b.claim_status = 'Rejected' 
            THEN 1 ELSE 0 END)                   AS rejected
    FROM billing b
    JOIN visits  v ON b.visit_id   = v.visit_id
    JOIN patients p ON v.patient_id = p.patient_id
    GROUP BY p.insurance_provider
    ORDER BY total_billed DESC
""", conn)

print("Insurance Billing Breakdown")
print(insurance_billing.to_string(index=False))

Insurance Billing Breakdown
insurance_provider  total_claims  total_billed  avg_billed  total_approved  paid  pending  rejected
         MediCareX          6532   134591163.0     20605.0     100135469.0  3875     1661       996
           CareOne          6283   130707993.0     20803.0      96997758.0  3787     1562       934
        HealthPlus          6220   130180741.0     20929.0      96251775.0  3680     1609       931
        SecureLife          5965   126289040.0     21172.0      93770886.0  3598     1431       936


In [16]:
# 10) Claim Rejection Analysis
# Which insurance providers reject the most claims?
rejection_analysis = pd.read_sql("""
    SELECT 
        p.insurance_provider,
        COUNT(b.bill_id)                              AS total_claims,
        SUM(CASE WHEN b.claim_status = 'Rejected' 
            THEN 1 ELSE 0 END)                        AS rejected_claims,
        ROUND(
            100.0 * SUM(CASE WHEN b.claim_status = 'Rejected' 
            THEN 1 ELSE 0 END) / COUNT(b.bill_id), 1) AS rejection_rate_pct
    FROM billing b
    JOIN visits   v ON b.visit_id   = v.visit_id
    JOIN patients p ON v.patient_id = p.patient_id
    GROUP BY p.insurance_provider
    ORDER BY rejection_rate_pct DESC
""", conn)

print("Claim Rejection Rate by Insurance Provider")
print(rejection_analysis.to_string(index=False))

Claim Rejection Rate by Insurance Provider
insurance_provider  total_claims  rejected_claims  rejection_rate_pct
        SecureLife          5965              936                15.7
         MediCareX          6532              996                15.2
        HealthPlus          6220              931                15.0
           CareOne          6283              934                14.9


In [17]:
# 11) Revenue Realization
# How much of what we bill actually gets approved?
revenue_realization = pd.read_sql("""
    SELECT 
        p.insurance_provider,
        ROUND(SUM(b.billed_amount), 0)                AS total_billed,
        ROUND(SUM(b.approved_amount), 0)              AS total_approved,
        ROUND(
            100.0 * SUM(b.approved_amount) / 
            SUM(b.billed_amount), 1)                  AS realization_rate_pct
    FROM billing b
    JOIN visits   v ON b.visit_id   = v.visit_id
    JOIN patients p ON v.patient_id = p.patient_id
    WHERE b.approved_amount IS NOT NULL
    GROUP BY p.insurance_provider
    ORDER BY realization_rate_pct DESC
""", conn)

print("Revenue Realization Rate by Insurance Provider")
print(revenue_realization.to_string(index=False))

Revenue Realization Rate by Insurance Provider
insurance_provider  total_billed  total_approved  realization_rate_pct
           CareOne   123701662.0      96997758.0                  78.4
        HealthPlus   122873457.0      96251775.0                  78.3
         MediCareX   128114983.0     100135469.0                  78.2
        SecureLife   120079039.0      93770886.0                  78.1


## Data Quality

In [18]:
# 12) Missing Records Check
print("=== MISSING VALUES CHECK ===\n")

for table in ['patients', 'visits', 'billing']:
    df = pd.read_sql(f"SELECT * FROM {table}", conn)
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) > 0:
        print(f"{table}:")
        print(nulls.to_string())
        print()
    else:
        print(f"{table}: ✓ No missing values\n")

=== MISSING VALUES CHECK ===

patients: ✓ No missing values

visits: ✓ No missing values

billing:
approved_amount    1318
payment_days        790



In [19]:
# 13) Duplicate Patients Check
duplicate_patients = pd.read_sql("""
    SELECT 
        patient_id,
        COUNT(*) AS occurrences
    FROM patients
    GROUP BY patient_id
    HAVING COUNT(*) > 1
""", conn)

if len(duplicate_patients) == 0:
    print("✓ No duplicate patient_ids found")
else:
    print(f"⚠ Found {len(duplicate_patients)} duplicate patient_ids")
    print(duplicate_patients)

✓ No duplicate patient_ids found


In [20]:
# 14) Orphan Records Check
# Visits without matching patients
orphan_visits = pd.read_sql("""
    SELECT COUNT(*) AS orphan_visits
    FROM visits v
    LEFT JOIN patients p ON v.patient_id = p.patient_id
    WHERE p.patient_id IS NULL
""", conn)

# Billing without matching visits
orphan_billing = pd.read_sql("""
    SELECT COUNT(*) AS orphan_billing
    FROM billing b
    LEFT JOIN visits v ON b.visit_id = v.visit_id
    WHERE v.visit_id IS NULL
""", conn)

print("Orphan visits  (no matching patient):", 
      orphan_visits['orphan_visits'][0])
print("Orphan billing (no matching visit)  :", 
      orphan_billing['orphan_billing'][0])

Orphan visits  (no matching patient): 0
Orphan billing (no matching visit)  : 0


In [21]:
# 15) Invalid LOS Check
invalid_los = pd.read_sql("""
    SELECT 
        COUNT(CASE WHEN length_of_stay_hours <= 0  THEN 1 END) AS zero_or_negative,
        COUNT(CASE WHEN length_of_stay_hours > 720 THEN 1 END) AS over_30_days,
        MIN(length_of_stay_hours)                              AS min_los,
        MAX(length_of_stay_hours)                             AS max_los,
        ROUND(AVG(length_of_stay_hours), 2)                   AS avg_los
    FROM visits
""", conn)

print("LOS Validation:")
print(invalid_los.to_string(index=False))

LOS Validation:
 zero_or_negative  over_30_days  min_los  max_los  avg_los
                0             0      0.5    78.42    19.55


In [22]:
# 16) Invalid Payment Days Check
invalid_payment = pd.read_sql("""
    SELECT 
        COUNT(CASE WHEN payment_days < 0   THEN 1 END) AS negative_days,
        COUNT(CASE WHEN payment_days > 365 THEN 1 END) AS over_1_year,
        COUNT(CASE WHEN payment_days IS NULL THEN 1 END) AS nulls,
        MIN(payment_days)                               AS min_days,
        MAX(payment_days)                               AS max_days
    FROM billing
""", conn)

print("Payment Days Validation:")
print(invalid_payment.to_string(index=False))

Payment Days Validation:
 negative_days  over_1_year  nulls  min_days  max_days
             0            0    790       1.0      55.0


## Export model_table.csv

In [23]:
# Join all three tables into one flat table for ML modeling.
# This is the single source of truth for Phase 2 EDA and Phase 3 Modeling.
model_table = pd.read_sql("""
    SELECT
        -- Patient features
        p.patient_id,
        p.age,
        p.gender,
        p.city,
        p.insurance_provider,
        p.chronic_flag,
        p.registration_date,

        -- Visit features
        v.visit_id,
        v.visit_date,
        v.department,
        v.visit_type,
        v.length_of_stay_hours,
        v.risk_score,
        v.doctor_id,

        -- Billing features
        b.bill_id,
        b.billed_amount,
        b.approved_amount,
        b.claim_status,
        b.payment_days,
        b.billing_date

    FROM visits v
    JOIN patients p ON v.patient_id = p.patient_id
    JOIN billing  b ON v.visit_id   = b.visit_id
    ORDER BY v.visit_date
""", conn)

print("model_table shape:", model_table.shape)
print("Columns:", model_table.columns.tolist())
model_table.head(3)

model_table shape: (25000, 20)
Columns: ['patient_id', 'age', 'gender', 'city', 'insurance_provider', 'chronic_flag', 'registration_date', 'visit_id', 'visit_date', 'department', 'visit_type', 'length_of_stay_hours', 'risk_score', 'doctor_id', 'bill_id', 'billed_amount', 'approved_amount', 'claim_status', 'payment_days', 'billing_date']


,patient_id,age,gender,city,insurance_provider,chronic_flag,registration_date,visit_id,visit_date,department,visit_type,length_of_stay_hours,risk_score,doctor_id,bill_id,billed_amount,approved_amount,claim_status,payment_days,billing_date
0,3993,9,M,Hyderabad,HealthPlus,1,2025-10-18,684,2025-01-20,General,ICU,5.79,Low,137,684,36721.68,36721.68,Paid,19.0,2025-08-15
1,76,59,M,Delhi,HealthPlus,0,2025-07-09,1412,2025-01-20,General,ER,34.80,Medium,109,1412,8365.47,4189.20,Pending,26.0,2025-11-14
2,3393,43,M,Hyderabad,HealthPlus,0,2025-07-10,1510,2025-01-20,ICU,ICU,31.37,Medium,135,1510,16529.35,0.00,Rejected,7.0,2025-04-24


In [24]:
# save the table
import os
os.makedirs("../outputs", exist_ok=True)
model_table.to_csv("../outputs/model_table.csv", index=False)
print(f"model_table.csv saved --> /outputs/model_table.csv")
print(f"Shape: {model_table.shape}")
print(f"Size: {os.path.getsize('../outputs/model_table.csv')/1024:.1f} KB")

model_table.csv saved --> /outputs/model_table.csv
Shape: (25000, 20)
Size: 3102.2 KB


In [25]:
# Close Connection
conn.close()
print("Database connection closed successfully")

Database connection closed successfully
